In [ ]:
import os
import numpy as np
from matplotlib import pyplot as plt
import torch
from torch import nn, optim
import torch.nn.functional as F
from preprocess_data import load_and_transform_MNIST_data_to_torch_tensor, print_torch, load_and_transform_custom_data_to_torch_tensor
import torchmetrics as tm

In [ ]:
# Load and transform the MNIST data to PyTorch tensors
train_dataloader, test_dataloader = load_and_transform_MNIST_data_to_torch_tensor()

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        # 1st conv layer
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=8, kernel_size=3, padding=1) # 'same' is padding=1 for k=3
        # 2nd conv layer
        self.conv2 = nn.Conv2d(in_channels=8, out_channels=16, kernel_size=3, padding=1) # 'same' is padding=2 for k=5
        # max pool layer
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        # fully connected layer
        self.fc1 = nn.Linear(16 * 7 * 7, 128)

    def forward(self, x):
        # Apply 1st convolution and ReLU activation
        x = F.relu(self.conv1(x))
        # Apply max pooling
        x = self.pool(x)
        # Apply 2nd convolution and ReLU activation
        x = F.relu(self.conv2(x))
        # Apply max pooling
        x = self.pool(x)
        # Flatten the tensor
        x = x.reshape(x.shape[0], -1)  
        # Apply fully connected layer
        x = self.fc1(x)            
        return x

In [ ]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(device)  
print(torch.backends.mps.is_available())
print(torch.backends.mps.is_built())

In [ ]:
def train_model(iterations, alpha, modelname):
    model = CNN().to(device)
    criterion = nn.CrossEntropyLoss()
    if os.path.exists("./models/"+modelname):
        print(f"Loading model {modelname}...")
        model.load_state_dict(
            torch.load("./models/" + modelname,
            map_location=device,
            weights_only=True
            )
        )
    else: 
        print(f"Creating new model {modelname}...")

    # Adam optimizer to update the weights of the model
    optimizer = optim.Adam(model.parameters(), lr=alpha)
    
    print(f"Training new model {modelname}...")
    for epoch in range(iterations):
        print(f"\nEpoch [{epoch + 1}/{iterations}]")

        model.train()

        correct = 0
        total = 0
        running_loss = 0

        for data, targets in train_dataloader:
            data = data.to(device)
            targets = targets.to(device)

            scores = model(data)
            loss = criterion(scores, targets)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # accuracy tracking
            _, preds = torch.max(scores, 1)
            correct += (preds == targets).sum().item()
            total += targets.size(0)
            running_loss += loss.item()

        acc = correct / total
        avg_loss = running_loss / len(train_dataloader)

        print(f"Loss: {avg_loss:.4f} | Accuracy: {acc:.4f}")

    torch.save(model.state_dict(), "./models/"+modelname)
    
def test_model(modelname):
    model = CNN().to(device)
    print(f"Loading model {modelname}...")
    model.load_state_dict(
        torch.load("./models/" + modelname,
        map_location=device,
        weights_only=True
        )
    )
    model.eval()
    correct = 0
    total = 0
    
    # calculate accuracy
    with torch.no_grad():
        for images, labels in test_dataloader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            probs = torch.softmax(outputs, dim=1)
            preds = torch.argmax(probs, dim=1)
            
            correct += (preds == labels).sum().item()
            total += labels.size(0)
        acc = correct / total

        # print random image 
        images, labels = next(iter(test_dataloader))
            
        images = images.to(device)
        labels = labels.to(device)

        idx = torch.randint(0, images.size(0), (1,)).item()
        img = images[idx].unsqueeze(0)   # keep shape [1,1,28,28]
        label = labels[idx]
        
        outputs = model(img)
        probs = torch.softmax(outputs, dim=1)   
        preds = torch.argmax(probs, dim=1)
        
        img_show = img.cpu().squeeze()
        plt.imshow(img_show)
        plt.axis("off")
        plt.show()

        print("Predictions:", preds.item(), "| True:", round(labels[idx].item()))
        print("Probabilities:", probs.cpu().numpy().round(2).tolist())
        print(f"Accuracy: {acc:.4f}")
        

def test_model_custom_data(modelname, image):
    model = CNN().to(device)
    print(f"Loading model {modelname}...")
    model.load_state_dict(
    torch.load("./models/" + modelname,
        map_location=device,
        weights_only=True)
    )    
    model.eval()
    
    activations = {}
    # ---- hook function ----
    def get_activation(name):
        def hook(model, input, output):
            activations[name] = output.detach()
        return hook

    model.conv1.register_forward_hook(get_activation("conv1"))
    model.conv1.register_forward_hook(get_activation("conv2"))    
    
    images = load_and_transform_custom_data_to_torch_tensor(image)
    images = images.to(device)
    
    with torch.no_grad():
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)
        preds = torch.argmax(probs, dim=1)

        print("Predictions:", preds.item())
        print("Probabilities:", probs.cpu().numpy().round(2).tolist())
        
        img = images.cpu().squeeze()
        plt.imshow(img)
        plt.axis("off")
        plt.show()
    
     # ---------------- CONV1 ----------------
    act1 = activations["conv1"][0].cpu()
    plot_feature_maps(act1, "Conv1 Activations")

    # ---------------- CONV2 ----------------
    act2 = activations["conv2"][0].cpu()
    plot_feature_maps(act2, "Conv2 Activations")


def plot_feature_maps(act, title):
    num_maps = min(16, act.shape[0])

    plt.figure(figsize=(6, 6))
    for i in range(num_maps):
        plt.subplot(4, 4, i + 1)
        plt.imshow(act[i], cmap="viridis")
        plt.axis("off")

    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

In [ ]:
train_model(200, 0.002, "model1_cnn")
for i in range (1,10):
    test_model_custom_data("model1_cnn", f"{i}.png")
# test_model("model1_cnn")